# 3. Feature Extraction

Extract features from the windowed datasets created by Notebook 02.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CUDB_PATH, VFDB_PATH, OUTPUT_DIR, CLIP_TOL
from src.features import extract_features_for_dataset
from src.io import load_signal

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
with open(CUDB_PATH / 'RECORDS') as f:
    records = [line.strip() for line in f if line.strip()]

cudb_set = set(records)


In [3]:
train_2s = pd.read_csv(OUTPUT_DIR / 'windows_2s_train.csv')
test_2s = pd.read_csv(OUTPUT_DIR / 'windows_2s_test.csv')

train_features_2s = extract_features_for_dataset( train_2s, cudb_set, CUDB_PATH, VFDB_PATH, clip_tol=CLIP_TOL)
test_features_2s = extract_features_for_dataset(
    test_2s, cudb_set, CUDB_PATH, VFDB_PATH, clip_tol=CLIP_TOL
)

train_features_2s.to_csv(OUTPUT_DIR / 'features_2s_train.csv', index=False)
test_features_2s.to_csv(OUTPUT_DIR / 'features_2s_test.csv', index=False)

print('Train features shape:', train_features_2s.shape)
print(train_features_2s.head())
print('\nTest features shape:', test_features_2s.shape)


Train features shape: (91608, 22)
   peak_to_peak       std       rms    zcr      skew  kurtosis  clip_fraction  \
0        3.5500  0.821209  0.821212  0.024 -0.437395  0.169426            0.0   
1        3.5025  0.809660  0.809663  0.026 -0.437384  0.161220            0.0   
2        3.5025  0.826251  0.827087  0.024 -0.399260 -0.051095            0.0   
3        3.5600  0.845127  0.845698  0.024 -0.363834 -0.124397            0.0   
4        3.5175  0.804078  0.805288  0.024 -0.336223  0.081178            0.0   

   fraction_imputed  used_hold_at_extreme  dominant_freq  ...  sample_entropy  \
0               0.0                   0.0            1.5  ...        0.114979   
1               0.0                   0.0            1.5  ...        0.117107   
2               0.0                   0.0            1.5  ...        0.107763   
3               0.0                   0.0            1.5  ...        0.104409   
4               0.0                   0.0            1.5  ...        0.095

In [8]:
# reconstruct which windows got dropped and why
train_2s_ids = set(zip(train_2s['record'].astype(str), train_2s['start'], train_2s['end']))

# simpler: just count how many rows had NaN originally vs how many were dropped
print("Original train window count:", len(train_2s))
print("Final train feature count:", len(train_features_2s))
print("Dropped:", len(train_2s) - len(train_features_2s))

Original train window count: 91609
Final train feature count: 91608
Dropped: 1


In [5]:
low_variance = train_features_2s[train_features_2s['std'] < 0.01]
print(f"Windows with std < 0.01: {len(low_variance)} out of {len(train_features_2s)} ({100*len(low_variance)/len(train_features_2s):.2f}%)")
print(low_variance['label'].value_counts())
print()
print(low_variance[['peak_to_peak', 'std', 'skew', 'kurtosis', 'record', 'label']].head(10))

Windows with std < 0.01: 24 out of 91608 (0.03%)
label
Transitional    16
Normal           8
Name: count, dtype: int64

       peak_to_peak           std      skew  kurtosis record         label
15144        0.0500  9.957816e-03 -0.664802 -0.262773   cu31        Normal
15145        0.0425  8.097685e-03 -0.611460  0.379898   cu31        Normal
15146        0.0425  8.375267e-03 -0.388979 -0.113723   cu31        Normal
15147        0.0425  8.495034e-03 -0.778235  0.294091   cu31        Normal
15148        0.0450  7.513814e-03 -0.348166  0.322582   cu31        Normal
15149        0.0450  9.406193e-03  0.014837 -0.510270   cu31        Normal
15167        0.0575  9.844606e-03  0.544611  0.719892   cu31        Normal
15168        0.0500  9.633924e-03  0.546260  0.616397   cu31        Normal
19092        0.0000  8.881784e-16  0.000000  0.000000   cu24  Transitional
19093        0.0000  8.881784e-16  0.000000  0.000000   cu24  Transitional


In [6]:
train_2s = pd.read_csv(OUTPUT_DIR / 'windows_2s_train.csv')
train_2s['record'] = train_2s['record'].astype(str)

cu24_windows = train_2s[train_2s['record'] == 'cu24'].copy()
signal, record_len = load_signal('cu24', CUDB_PATH)

flat_windows = []
for _, row in cu24_windows.iterrows():
    w = signal[row['start']:row['end']]
    w_valid = w[~np.isnan(w)]
    if len(w_valid) > 10 and w_valid.std() < 1e-10:
        flat_windows.append((row['start'], row['end'], row['label']))

print(f"Found {len(flat_windows)} flat windows in cu24")
for s, e, label in flat_windows:
    print(f"  samples {s}-{e} ({s/250:.1f}s-{e/250:.1f}s), label={label}")
    print(f"  raw values: {signal[s:e][:20]}")

Found 9 flat windows in cu24
  samples 106125-106625 (424.5s-426.5s), label=Transitional
  raw values: [5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175
 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175]
  samples 106250-106750 (425.0s-427.0s), label=Transitional
  raw values: [5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175
 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175]
  samples 106375-106875 (425.5s-427.5s), label=Transitional
  raw values: [5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175
 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175]
  samples 109000-109500 (436.0s-438.0s), label=Transitional
  raw values: [5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175
 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175 5.1175]
  samples 110625-111125 (442.5s-444.5s), label=Transitional
  raw values: [5.1175 5.1175 5.1175 5.1175 

In [7]:
print("Train:", (train_features_2s['std'] < 1e-8).sum(), "out of", len(train_features_2s))
print("Test:", (test_features_2s['std'] < 1e-8).sum(), "out of", len(test_features_2s))

Train: 14 out of 91608
Test: 0 out of 23914
